In [ ]:
!pip -q install 'voxcpm>=0.2' soundfile huggingface_hub pillow numpy
!which ffmpeg >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq ffmpeg)
import os, subprocess, pathlib
print('GPU check:')
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=False)


In [ ]:
from huggingface_hub import snapshot_download
import os
MODEL_DIR = '/kaggle/working/VoxCPM2'
if not os.path.isfile(os.path.join(MODEL_DIR,'model.safetensors')):
    snapshot_download(repo_id='openbmb/VoxCPM2',local_dir=MODEL_DIR)
print('VoxCPM2 model available',MODEL_DIR)


In [ ]:
import os, math, re, json, pathlib, subprocess, wave
import numpy as np
import soundfile as sf
from voxcpm import VoxCPM
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance

SCENES=[
    "Every life has a final heartbeat. But what actually happens when a person dies?",
    "When the heart stops pumping, oxygen-rich blood stops reaching the brain.",
    "Consciousness can be lost within seconds. But cardiac arrest is not always irreversible. Emergency CPR can save lives.",
    "If circulation cannot be restored, the brain and other organs lose the oxygen they need. Eventually, their functions stop.",
    "After death, the body cools. Gravity causes blood to settle. Later, muscles can gradually become stiff.",
    "Some people who survive cardiac arrest describe vivid near-death experiences. Those reports matter, but they do not establish an afterlife.",
    "Research on brain activity near death raises important questions. Electrical signals alone do not tell us whether someone is conscious.",
    "Science explains many changes in the body. What, if anything, happens to awareness after irreversible death remains unknown."
]
DESIGN="An adult English science documentary narrator, warm low-mid voice, controlled suspense, empathetic, clear pronunciation, measured pacing, natural breath, never theatrical or scary."
SRC=pathlib.Path('/kaggle/input/e2micro-death-documentary-scenes-20261103')
OUT=pathlib.Path('/kaggle/working/death_video')
OUT.mkdir(parents=True,exist_ok=True)
images=sorted(SRC.glob('scene_*.jpg'))
assert len(images)==len(SCENES),(len(images),str(SRC))
print('Images',len(images))
model=VoxCPM.from_pretrained(MODEL_DIR,load_denoiser=False,optimize=False,device='cuda')
try: SR=int(model.tts_model.sample_rate)
except Exception: SR=48000
clips=[]; log=[]
for i,text in enumerate(SCENES):
    print('VoxCPM2 narration scene',i+1,'/',len(SCENES),flush=True)
    speech=model.generate(text='('+DESIGN+')'+text,cfg_value=2.0,inference_timesteps=10,retry_badcase=True,normalize=False)
    speech=np.asarray(speech,dtype=np.float32).reshape(-1)
    if not len(speech): raise RuntimeError('VoxCPM2 returned empty speech for scene '+str(i))
    speech=np.nan_to_num(speech)
    speech=np.clip(speech,-1,1)
    # Minimal pause at scene boundary
    audio=np.concatenate([speech,np.zeros(int(.18*SR),dtype=np.float32)])
    name=OUT/f'speech_{i:02d}.wav'
    sf.write(name,audio,SR,subtype='PCM_16')
    clips.append(audio)
    log.append({'scene':i+1,'duration':round(len(audio)/SR,3),'text':text})
    print('Seconds',round(len(audio)/SR,2),flush=True)
del model
joined=np.concatenate(clips)
sf.write(OUT/'narration.wav',joined,SR,subtype='PCM_16')
(OUT/'script.json').write_text(json.dumps(log,indent=2))
print('TOTAL NARRATION SEC',len(joined)/SR,flush=True)


In [ ]:
import math, textwrap
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from pathlib import Path
W,H,FPS=720,1280,24
font_path='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
bold_path='/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
titlefont=ImageFont.truetype(bold_path,37)
bodyfont=ImageFont.truetype(bold_path,32)
smallfont=ImageFont.truetype(font_path,20)
def fit_cover(img, dims):
    w,h=dims
    f=max(w/img.width,h/img.height)
    im=img.resize((math.ceil(img.width*f),math.ceil(img.height*f)),Image.Resampling.LANCZOS)
    x=(im.width-w)//2;y=(im.height-h)//2
    return im.crop((x,y,x+w,y+h))
def wrapped_text(d,text,font,max_width):
    words=text.split();lines=[];line=''
    for word in words:
        candidate=(line+' '+word).strip()
        if d.textbbox((0,0),candidate,font=font)[2] > max_width and line:
            lines.append(line);line=word
        else:line=candidate
    if line:lines.append(line)
    return lines

for i,(p,script) in enumerate(zip(images,SCENES)):
    orig=Image.open(p).convert('RGB')
    bg=fit_cover(orig,(W,H)).filter(ImageFilter.GaussianBlur(32))
    tint=Image.new('RGBA',(W,H),(5,11,25,100))
    base=Image.alpha_composite(bg.convert('RGBA'),tint)
    focus=orig.resize((680,680),Image.Resampling.LANCZOS).convert('RGBA')
    mask=Image.new('L',(680,680),0);m=ImageDraw.Draw(mask)
    m.rounded_rectangle((0,0,680,680),radius=22,fill=255)
    base.paste(focus,(20,230),mask)
    v=Image.new('RGBA',(W,H),(0,0,0,0))
    vd=ImageDraw.Draw(v,'RGBA')
    for j in range(0,300,3):
        vd.rectangle((0,H-j-3,W,H-j),fill=(1,5,17,int(220*j/300)))
    base=Image.alpha_composite(base,v)
    d=ImageDraw.Draw(base)
    d.text((35,48),'WHAT HAPPENS',font=titlefont,fill=(245,243,232))
    d.text((35,98),'WHEN WE DIE?',font=titlefont,fill=(242,207,129))
    d.text((38,180),f'SCIENCE DOCUMENTARY   •   {i+1:02d}/08',font=smallfont,fill=(185,202,215))
    lines=wrapped_text(d,script,bodyfont,W-86)
    # Long captions are positioned using measured height, to stay above the safe bottom
    lineheight=42
    if len(lines)>6:
        lines=wrapped_text(d,script,ImageFont.truetype(bold_path,28),W-86)
        usedfont=ImageFont.truetype(bold_path,28)
        lineheight=38
    else: usedfont=bodyfont
    y0=1040-min(len(lines),6)*lineheight//2
    for j,line in enumerate(lines):
        bbox=d.textbbox((0,0),line,font=usedfont)
        d.text(((W-(bbox[2]-bbox[0]))//2,y0+j*lineheight),line,font=usedfont,fill=(250,248,242),stroke_width=1,stroke_fill=(10,20,27))
    d.text((W//2-130,1217),'E2MICRO  •  VOXCPM2',font=smallfont,fill=(183,194,206))
    base.convert('RGB').save(OUT/f'card_{i:02d}.jpg',quality=92)
print('Created',len(SCENES),'captioned scene cards')


In [ ]:
import subprocess, math, pathlib
frames=[];FPS=24
for i,row in enumerate(log):
    dur=row['duration']
    n=max(30,math.ceil(dur*FPS))
    t=n/FPS
    filt=(f"scale=760:1352,zoompan=z='min(zoom+0.00009,1.06)':"
          f"x='(iw-iw/zoom)/2':y='(ih-ih/zoom)/2':d={n}:"
          f"s=720x1280:fps={FPS},format=yuv420p,"
          f"fade=t=in:st=0:d=0.23,fade=t=out:st={max(.01,t-.25):.3f}:d=0.23")
    dest=OUT/f'clip_{i:02d}.mp4'
    cmd=['ffmpeg','-hide_banner','-loglevel','error','-y',
         '-loop','1','-i',str(OUT/f'card_{i:02d}.jpg'),
         '-vf',filt,'-frames:v',str(n),'-an',
         '-c:v','libx264','-preset','veryfast','-crf','22',
         '-pix_fmt','yuv420p','-r',str(FPS),str(dest)]
    print('Encoding video scene',i+1,'seconds',round(t,1),flush=True)
    subprocess.run(cmd,check=True)
    frames.append(dest)
listfile=OUT/'clips.txt'
listfile.write_text(''.join("file '"+str(p)+"'\n" for p in frames))
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y',
                '-f','concat','-safe','0','-i',str(listfile),
                '-c','copy',str(OUT/'silent.mp4')],check=True)
print('Silent video ready',flush=True)


In [ ]:
import numpy as np, soundfile as sf, subprocess, json
n=len(joined)
tt=np.arange(n,dtype=np.float64)/SR
# Original quiet three-chord ambient underscore, no copyrighted material
music=np.zeros(n,dtype=np.float64)
total=tt[-1]
for i,notes in enumerate([(146.83,196,246.94),(174.61,220,261.63),(130.81,164.81,196)]):
    lo=i*total/3;hi=(i+1)*total/3
    envelope=np.clip((tt-lo)/1.8,0,1)*np.clip((hi-tt)/1.9,0,1)
    for j,f in enumerate(notes):
        music += .021/(j+1)*envelope*np.sin(2*np.pi*f*tt)
    music += .002*envelope*np.sin(2*np.pi*55*tt)
music*=np.clip(tt/1.3,0,1)*np.clip((total-tt)/1.2,0,1)
# Keep narration clearly audible above the soundtrack
mix=np.clip(joined.astype(np.float64)*.92+music,-.98,.98).astype('float32')
sf.write(OUT/'master.wav',mix,SR,subtype='PCM_16')
final=OUT/'what_happens_when_we_die.mp4'
subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y',
   '-i',str(OUT/'silent.mp4'),'-i',str(OUT/'master.wav'),
   '-c:v','copy','-c:a','aac','-b:a','160k','-ar','48000','-shortest',
   '-movflags','+faststart',str(final)],check=True)
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size',
  '-of','json',str(final)],capture_output=True,text=True,check=True)
print('COMPLETE',final,probe.stdout,flush=True)
(OUT/'sources.txt').write_text('Sources: CDC cardiac arrest https://www.cdc.gov/heart-disease/about/cardiac-arrest.html\nCleveland Clinic https://my.clevelandclinic.org/health/articles/23144-what-happens-when-you-die\nCanada clinical guideline https://pmc.ncbi.nlm.nih.gov/articles/PMC10203028/\n')
